# Unión GeoJSON × resultados electorales — Cárdenas, Tabasco

Une los CSV de sección (salida del notebook 01) con el GeoJSON de secciones electorales  
para producir tres archivos GeoJSON listos para el geovisor:

- `resultados_seccion_gubernatura.geojson`
- `resultados_seccion_diputaciones.geojson`
- `resultados_seccion_ayuntamiento.geojson`

In [1]:
import json
import csv
from pathlib import Path
from copy import deepcopy

In [2]:
GEOJSON_SECCIONES = Path("../datos/insumos/geojson/03_seccion.geojson")
PROCESADOS        = Path("../datos/procesados")

TIPOS = ["gubernatura", "diputaciones", "ayuntamiento"]

## 1. Cargar CSVs de resultados por sección

In [3]:
def leer_csv(tipo):
    """
    Lee el CSV de resultados por sección para un tipo de elección.
    Retorna dict { int(seccion): {col: val, ...} }.
    """
    ruta = PROCESADOS / f"resultados_seccion_{tipo}.csv"
    datos = {}
    with open(ruta, encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            sec = int(row["seccion"])   # clave numérica para comparar con GeoJSON
            datos[sec] = row
    return datos


resultados = {tipo: leer_csv(tipo) for tipo in TIPOS}

for tipo, d in resultados.items():
    print(f"{tipo:<15}  {len(d):>3} secciones  (incl. sec 0 = voto anticipado)")

gubernatura      159 secciones  (incl. sec 0 = voto anticipado)
diputaciones     159 secciones  (incl. sec 0 = voto anticipado)
ayuntamiento     123 secciones  (incl. sec 0 = voto anticipado)


## 2. Cargar GeoJSON de secciones electorales

In [4]:
with open(GEOJSON_SECCIONES, encoding="utf-8") as f:
    geojson_base = json.load(f)

print(f"Features totales en GeoJSON: {len(geojson_base['features'])}")
print(f"Propiedades disponibles: {list(geojson_base['features'][0]['properties'].keys())}")

Features totales en GeoJSON: 1185
Propiedades disponibles: ['ID', 'ENTIDAD', 'DISTRITO_F', 'DISTRITO_L', 'MUNICIPIO', 'SECCION', 'TIPO', 'CONTROL', 'GEOMETRY1_', 'CIRCUITO', 'DISTRITO_J']


## 3. Diagnóstico del join

Clave de unión: `SECCION` (entero en GeoJSON) ↔ `seccion` (int del CSV).  
Se esperan ~158 matches por tipo (la sección `0000` = voto anticipado no tiene geometría).

In [5]:
secciones_geojson = {f["properties"]["SECCION"] for f in geojson_base["features"]}

for tipo, datos in resultados.items():
    secs_csv   = set(datos.keys())
    con_match  = secs_csv & secciones_geojson
    sin_match  = secs_csv - secciones_geojson
    print(f"{tipo:<15}  match={len(con_match):>3}  sin geometría={sorted(sin_match)}")

gubernatura      match=158  sin geometría=[0]
diputaciones     match=158  sin geometría=[0]
ayuntamiento     match=122  sin geometría=[0]


## 4. Función de unión y exportación

In [6]:
# Columnas del CSV que se convierten a float (votos, porcentajes);
# el resto permanece como string.
COLS_NUMERICAS = {
    "PAN", "PRI", "PRD", "PVEM", "PT", "MC", "MORENA", "CAND_IND",
    "total_pvem_pt_morena", "total_pvem_morena", "total_pan_pri",
    "votos_validos", "cand_no_reg", "votos_nulos", "total_votos",
    "lista_nominal", "participacion",
}


def castear(col, val):
    """Convierte strings vacíos a None y columnas numéricas a float."""
    if val == "" or val is None:
        return None
    if col in COLS_NUMERICAS:
        try:
            return float(val)
        except ValueError:
            return None
    return val


def construir_geojson(tipo, datos_csv):
    """
    Filtra el GeoJSON base a las secciones con datos electorales para `tipo`
    y añade las propiedades del CSV a cada feature.

    Retorna un dict FeatureCollection listo para serializar.
    """
    features_out = []

    for feat in geojson_base["features"]:
        sec_num = feat["properties"]["SECCION"]
        if sec_num not in datos_csv:
            continue  # sección no pertenece a Cárdenas / este tipo

        # Copia profunda para no mutar el original
        nueva_feat = deepcopy(feat)

        # Añadir propiedades electorales al feature
        for col, val in datos_csv[sec_num].items():
            nueva_feat["properties"][col] = castear(col, val)

        features_out.append(nueva_feat)

    return {
        "type": "FeatureCollection",
        "features": features_out,
    }

## 5. Generar y exportar los tres GeoJSONs

In [7]:
for tipo in TIPOS:
    gj = construir_geojson(tipo, resultados[tipo])
    salida = PROCESADOS / f"resultados_seccion_{tipo}.geojson"

    with open(salida, "w", encoding="utf-8") as f:
        json.dump(gj, f, ensure_ascii=False, separators=(",", ":"))

    size_kb = salida.stat().st_size / 1024
    print(f"{salida.name:<45}  {len(gj['features']):>3} features  {size_kb:>7.1f} KB")

resultados_seccion_gubernatura.geojson         158 features    880.4 KB
resultados_seccion_diputaciones.geojson        158 features    880.7 KB
resultados_seccion_ayuntamiento.geojson        122 features    639.8 KB


## 6. Verificación rápida

In [8]:
# Mostrar propiedades del primer feature de gubernatura
with open(PROCESADOS / "resultados_seccion_gubernatura.geojson", encoding="utf-8") as f:
    gj_check = json.load(f)

feat0 = gj_check["features"][0]
print("Sección:", feat0["properties"]["SECCION"])
print("Municipio GeoJSON:", feat0["properties"]["MUNICIPIO"])
print()
print("Propiedades electorales:")
for k, v in feat0["properties"].items():
    if k not in ("ID", "ENTIDAD", "DISTRITO_F", "DISTRITO_L", "MUNICIPIO",
                 "TIPO", "CONTROL", "GEOMETRY1_", "CIRCUITO", "DISTRITO_J"):
        print(f"  {k:<25} {v}")

Sección: 46
Municipio GeoJSON: 2

Propiedades electorales:
  SECCION                   46
  seccion                   0046
  tipo_eleccion             gubernatura
  distrito                  DT-02
  PAN                       3.0
  PRI                       7.0
  PRD                       57.0
  PVEM                      26.0
  PT                        20.0
  MC                        16.0
  MORENA                    372.0
  CAND_IND                  None
  total_pvem_pt_morena      457.0
  total_pvem_morena         None
  total_pan_pri             10.0
  votos_validos             540.0
  cand_no_reg               0.0
  votos_nulos               14.0
  total_votos               554.0
  lista_nominal             1140.0
  participacion             0.48596491228070177


In [9]:
# Verificar que participacion está dentro de un rango razonable
for tipo in TIPOS:
    with open(PROCESADOS / f"resultados_seccion_{tipo}.geojson", encoding="utf-8") as f:
        gj = json.load(f)
    partics = [feat["properties"].get("participacion") for feat in gj["features"]]
    partics = [p for p in partics if p is not None]
    print(f"{tipo:<15}  features={len(gj['features']):>3}  "
          f"participacion min={min(partics):.3f}  max={max(partics):.3f}")

gubernatura      features=158  participacion min=0.307  max=1.331
diputaciones     features=158  participacion min=0.397  max=1.024
ayuntamiento     features=122  participacion min=0.384  max=0.934
